# dYdX Perpetual Futures Data Import

This notebook imports:
1. **OHLC data** (prices and volume) for dYdX perpetual futures
2. **Funding rate data** (hourly funding rates)

## About dYdX
- **Type**: Cosmos-based blockchain (dYdX v4, launched 2023)
- **Available**: Globally (no KYC, no geo-restrictions)
- **Contracts**: 100+ perpetual futures
- **Settlement**: USDC (cross-margined, not isolated)
- **Funding**: Hourly (charged at XX:00:00 UTC)

## Funding Rate Details
- **Formula**: Funding Rate = (Premium Component / 8) + Interest Rate
- **Frequency**: Charged every hour (not every 8 hours!)
- **Premium**: TWAP of mark-index spread over last 60 minutes
- **Interest Rate**: 0% for cross markets, 0.125 bps/hour for isolated

## API Details
- **Endpoint**: https://indexer.dydx.trade/v4
- **Ticker Format**: Hyphenated (BTC-USD, ETH-USD)
- **Timezone**: UTC
- **Limit**: ~1000 candles/records per request (pagination implemented automatically)

In [ ]:
from clients.dydx_client import DydxClient
from clients.db_client import DBClient
import pandas as pd
from datetime import datetime

In [ ]:
# Configuration
EXCHANGE = 'dydx'
INTERVAL = '1h'  # 1 hour candles
START_DATE = '2025-01-01'  # dYdX limits to 100 candles per request
END_DATE = datetime.today().strftime('%Y-%m-%d')
# Test with Bitcoin perpetual
test_symbol = 'BTC-USD'
dydx = DydxClient()

## Test Download - Single Contract

In [ ]:


# Test OHLC download
df_ohlc = dydx.download_ohlc(
    symbol=test_symbol,
    interval=INTERVAL,
    start_date=START_DATE,
    end_date=END_DATE
)

print(f"\nOHLC Data Preview:")
print(df_ohlc.head())
print(f"\nShape: {df_ohlc.shape}")
print(f"Date range: {df_ohlc['timestamp'].min()} to {df_ohlc['timestamp'].max()}")

In [ ]:
# Test funding rate download
df_funding = dydx.download_funding_rates(
    symbol=test_symbol,
    start_date=START_DATE,
    end_date=END_DATE
)

print(f"\nFunding Rate Data Preview:")
print(df_funding.head())
print(f"Date range: {df_funding['timestamp'].min()} to {df_funding['timestamp'].max()}")

## Import OHLC data for all dYdX perpetual futures

In [ ]:
with DBClient() as db:
    instruments_dict = db.get_perpetuals_dict(EXCHANGE)

print(f"Found {len(instruments_dict)} dYdX instruments in database")

In [ ]:
with DBClient() as db:
    for ticker, instrument_id in instruments_dict.items():
        print(f"\nProcessing {ticker}...")
        
        # Download OHLC data
        df_ohlc = dydx.download_ohlc(
            symbol=ticker,
            interval=INTERVAL,
            start_date=START_DATE,
            end_date=END_DATE
        )
        
        if df_ohlc.empty:
            print(f"No OHLC data available for {ticker}")
            continue
        
        # Check existing data range
        min_ts, max_ts = db.get_timestamp_range(instrument_id)
        
        # Filter out duplicates
        if min_ts and max_ts:
            df_ohlc = df_ohlc[(df_ohlc['timestamp'] < min_ts) | (df_ohlc['timestamp'] > max_ts)]
            
            if df_ohlc.empty:
                print(f"All data already exists, skipping")
                continue
        
        # Add instrument_id and insert
        df_ohlc['instrument_id'] = instrument_id
        rows_inserted = db.insert_market_data(df_ohlc)
        
        if rows_inserted > 0:
            print(f"Inserted {rows_inserted:,} OHLC records")
        else:
            print(f"No new OHLC records inserted")

print("OHLC IMPORT COMPLETED")

## Import funding rate data for all dYdX perpetual futures

In [ ]:
with DBClient() as db:
    instruments_dict = db.get_perpetuals_dict(EXCHANGE)
    
    for ticker, instrument_id in instruments_dict.items():
        print(f"\nProcessing funding for {ticker}...")
        
        # Download funding rate data
        df_funding = dydx.download_funding_rates(
            symbol=ticker,
            start_date=START_DATE,
            end_date=END_DATE
        )
        
        if df_funding.empty:
            print(f"No funding data available for {ticker}")
            continue
        
        # Check existing funding data range
        min_ts, max_ts = db.get_funding_timestamp_range(instrument_id)
        
        # Filter out duplicates
        if min_ts and max_ts:
            df_funding = df_funding[(df_funding['timestamp'] < min_ts) | (df_funding['timestamp'] > max_ts)]
            
            if df_funding.empty:
                print(f"All funding data already exists, skipping")
                continue
        
        # Add instrument_id and insert
        df_funding['instrument_id'] = instrument_id
        rows_inserted = db.insert_funding_data(df_funding)
        
        if rows_inserted > 0:
            print(f"Inserted {rows_inserted:,} funding records")
        else:
            print(f"No new funding records inserted")

print("FUNDING RATE IMPORT COMPLETED")